# BTS July 2026 data cleaning and audit

This notebook makes the dashboard's data definitions inspectable. It keeps the raw source separate, reports data-quality issues before transformation, independently reconciles source totals to the aggregated dashboard data, and only overwrites `dist/data.json` after an explicit approval switch.

Official source: [BTS Reporting Carrier On-Time Performance](https://www.transtats.bts.gov/DL_SelectFields.aspx?QO_fu146_anzr=b0-gvzr&gnoyr_VQ=FGJ), July 2026. The raw ZIP remains unchanged in `raw_data/`.

## 1. Setup

Run the notebook from top to bottom. It needs Python 3, pandas, and a Jupyter kernel. In VS Code, select a Python kernel and install missing packages when prompted.

In [ ]:
from collections import Counter
from pathlib import Path
import json
import os
import zipfile

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)

candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd() / "bts-delay-dashboard"]
PROJECT_ROOT = next(
    (path.resolve() for path in candidate_roots if (path / "scripts" / "build_data.py").exists()),
    None,
)
assert PROJECT_ROOT is not None, "Open this notebook from the bts-delay-dashboard project."

default_zip = PROJECT_ROOT / "raw_data" / "On_Time_Data_202607.zip"
RAW_ZIP = Path(os.environ.get("BTS_RAW_ZIP", default_zip)).expanduser().resolve()
OUTPUT_JSON = PROJECT_ROOT / "dist" / "data.json"
TOP_AIRPORTS = 0  # 0 retains all origin airports

assert RAW_ZIP.exists(), f"Raw BTS ZIP not found: {RAW_ZIP}"
print(f"Project: {PROJECT_ROOT}")
print(f"Raw source: {RAW_ZIP}")
print(f"Dashboard data: {OUTPUT_JSON}")

## 2. Inspect the source without changing it

The ZIP is read directly. No extraction or modification of the official file is required.

In [ ]:
with zipfile.ZipFile(RAW_ZIP) as archive:
    csv_members = [name for name in archive.namelist() if name.lower().endswith(".csv")]

assert csv_members, "The ZIP contains no CSV file."
assert len(csv_members) == 1, f"Expected one CSV, found {len(csv_members)}: {csv_members}"
CSV_MEMBER = csv_members[0]
print(f"CSV inside ZIP: {CSV_MEMBER}")

In [ ]:
required_columns = [
    "FlightDate", "DayOfWeek", "CRSDepTime",
    "Reporting_Airline", "Origin", "OriginCityName",
    "Cancelled", "Diverted", "ArrDelayMinutes", "ArrDel15",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay", "Distance",
]
audit_key_columns = [
    "Flight_Number_Reporting_Airline", "Dest"
]

with zipfile.ZipFile(RAW_ZIP) as archive:
    with archive.open(CSV_MEMBER) as source:
        source_header = pd.read_csv(source, nrows=0).columns.tolist()

missing_columns = sorted(set(required_columns) - set(source_header))
assert not missing_columns, f"Required columns missing: {missing_columns}"
load_columns = required_columns + [c for c in audit_key_columns if c in source_header]
text_columns = [
    "FlightDate", "CRSDepTime", "Reporting_Airline", "Origin",
    "OriginCityName", "Flight_Number_Reporting_Airline", "Dest", "CRSDepTime",
]
text_dtypes = {column: "string" for column in text_columns if column in load_columns}

with zipfile.ZipFile(RAW_ZIP) as archive:
    with archive.open(CSV_MEMBER) as source:
        raw = pd.read_csv(
            source, usecols=load_columns, dtype=text_dtypes, low_memory=False
        )

print(f"Rows: {len(raw):,}")
print(f"Loaded audit columns: {len(raw.columns)} of {len(source_header)} source columns")
print(raw.head(3).to_string(index=False))

## 3. Missing values and type conversion audit

Missing values are reported before any fill operation. Numeric conversion failures are counted instead of silently becoming zero. Delay-cause fields are normally blank for flights delayed less than 15 minutes; the later reconciliation check separately identifies delayed flights that should have attribution but do not.

In [ ]:
missingness = pd.DataFrame({
    "missing_rows": raw[required_columns].isna().sum(),
    "missing_percent": raw[required_columns].isna().mean().mul(100).round(2),
}).sort_values(["missing_rows", "missing_percent"], ascending=False)
print(missingness.to_string())

In [ ]:
df = raw.copy()
for column in text_columns:
    if column in df.columns:
        df[column] = df[column].astype("string").str.strip().replace("", pd.NA)

numeric_columns = [
    "DayOfWeek", "Cancelled", "Diverted", "ArrDelayMinutes", "ArrDel15",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay", "Distance",
]
conversion_rows = []
for column in numeric_columns:
    converted = pd.to_numeric(df[column], errors="coerce")
    failed = df[column].notna() & converted.isna()
    conversion_rows.append({"column": column, "conversion_failures": int(failed.sum())})
    df[column] = converted

df["FlightDateParsed"] = pd.to_datetime(df["FlightDate"], errors="coerce")
conversion_audit = pd.DataFrame(conversion_rows)
print(conversion_audit.to_string(index=False))

## 4. Row-level validity checks

`failed_rows` is the number of records requiring attention. Error checks affect metric validity. Warning checks may be legitimate but should be explained.

In [ ]:
audit_checks = []

def add_check(name, failed, severity, explanation):
    count = int(failed.sum()) if hasattr(failed, "sum") else int(failed)
    audit_checks.append({
        "check": name,
        "severity": severity,
        "failed_rows": count,
        "status": "PASS" if count == 0 else "REVIEW",
        "explanation": explanation,
    })

date_missing = df["FlightDateParsed"].isna()
date_outside_month = df["FlightDateParsed"].notna() & ~df["FlightDateParsed"].between(
    pd.Timestamp("2026-07-01"), pd.Timestamp("2026-07-31")
)
add_check("FlightDate parses", date_missing, "error", "Every record needs a valid date.")
add_check("FlightDate is in July 2026", date_outside_month, "error", "This dashboard represents one released month.")

expected_july_dates = pd.date_range("2026-07-01", "2026-07-31", freq="D")
observed_july_dates = pd.DatetimeIndex(
    df["FlightDateParsed"].dropna().dt.normalize().unique()
).sort_values()
missing_july_dates = expected_july_dates.difference(observed_july_dates)
daily_flight_counts = (
    df.groupby("FlightDateParsed").size()
    .reindex(expected_july_dates, fill_value=0)
    .rename("flight_count")
)
add_check(
    "All 31 July dates are represented",
    len(missing_july_dates),
    "error",
    "The exception count is missing calendar days; the audit does not impute flight records.",
)
print(f"Calendar coverage: {len(observed_july_dates)}/31 July dates observed")
print("Missing July dates:", [date.strftime("%Y-%m-%d") for date in missing_july_dates] or "None")
print("Flights reported on each July date:")
print(daily_flight_counts.to_string())

expected_weekday = df["FlightDateParsed"].dt.dayofweek.add(1)
weekday_mismatch = df["DayOfWeek"].notna() & expected_weekday.notna() & df["DayOfWeek"].ne(expected_weekday)
add_check("DayOfWeek matches FlightDate", weekday_mismatch, "error", "BTS uses Monday=1 through Sunday=7.")

for column in ["Cancelled", "Diverted"]:
    add_check(f"{column} is present", df[column].isna(), "error", "The status denominator requires a binary flag.")
    add_check(f"{column} is 0 or 1", df[column].notna() & ~df[column].isin([0, 1]), "error", "Unexpected flag values change record classification.")

cancelled = df["Cancelled"].eq(1)
diverted = df["Diverted"].eq(1)
add_check("A flight is not both cancelled and diverted", cancelled & diverted, "error", "These outcomes should be mutually exclusive.")

non_cancelled_or_diverted = ~(cancelled | diverted)
completed = non_cancelled_or_diverted & df["ArrDelayMinutes"].notna()
missing_completed_delay = non_cancelled_or_diverted & df["ArrDelayMinutes"].isna()
add_check("Non-cancelled/non-diverted flights have arrival outcome", missing_completed_delay, "error", "Otherwise the flight cannot enter the on-time denominator.")
add_check("ArrDelayMinutes is non-negative", df["ArrDelayMinutes"].notna() & df["ArrDelayMinutes"].lt(0), "error", "This BTS field sets early arrivals to zero.")
add_check("Completed flights have ArrDel15", completed & df["ArrDel15"].isna(), "error", "The 15-minute delay indicator defines the on-time rate.")
add_check("ArrDel15 is 0 or 1 when present", df["ArrDel15"].notna() & ~df["ArrDel15"].isin([0, 1]), "error", "Unexpected indicator values change the delayed count.")
delay_flag_mismatch = completed & df["ArrDel15"].notna() & df["ArrDel15"].eq(1).ne(df["ArrDelayMinutes"].ge(15))
add_check("ArrDel15 agrees with ArrDelayMinutes >= 15", delay_flag_mismatch, "error", "This independently checks the dashboard's on-time definition.")

expected_blocks = {"0001-0559"} | {f"{hour:02d}00-{hour:02d}59" for hour in range(6, 24)}
bad_time_block = df["DepTimeBlk"].isna() | ~df["DepTimeBlk"].isin(expected_blocks)
add_check("Scheduled departure block is recognized", bad_time_block, "error", "The heatmap requires the published BTS time blocks.")
add_check("Distance is positive", df["Distance"].isna() | df["Distance"].le(0), "error", "Distance is a quantitative field and should be positive.")

cause_columns = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
for column in cause_columns:
    add_check(f"{column} is non-negative", df[column].notna() & df[column].lt(0), "error", "Delay-cause minutes cannot be negative.")

cause_sum = df[cause_columns].fillna(0).sum(axis=1)
attributed_delay = completed & df["ArrDel15"].eq(1)
cause_mismatch = attributed_delay & cause_sum.sub(df["ArrDelayMinutes"]).abs().gt(0.01)
add_check("Delay causes reconcile to arrival delay for 15+ minute delays", cause_mismatch, "warning", "BTS attribution can expose source revisions or field-definition differences.")

candidate_key = [c for c in ["FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline", "Origin", "Dest", "CRSDepTime"] if c in df.columns]
candidate_duplicate_rows = df.duplicated(candidate_key, keep=False) if len(candidate_key) == 6 else pd.Series(False, index=df.index)
add_check("Candidate flight key is unique", candidate_duplicate_rows, "warning", f"Candidate key: {candidate_key}. Review duplicates before removing any rows.")

audit_table = pd.DataFrame(audit_checks)
print(audit_table.to_string(index=False))

In [ ]:
issues = audit_table.loc[audit_table["failed_rows"].gt(0)]
print(f"Checks passed: {(audit_table['failed_rows'] == 0).sum()} / {len(audit_table)}")
if issues.empty:
    print("No row-level exceptions were found.")
else:
    print("Checks requiring review:")
    print(issues[["check", "severity", "failed_rows", "explanation"]].to_string(index=False))

if cause_mismatch.any():
    cause_review_columns = [
        "FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline",
        "Origin", "Dest", "ArrDelayMinutes", *cause_columns,
    ]
    cause_review = df.loc[cause_mismatch, cause_review_columns].copy()
    cause_review["reportedCauseMinutes"] = cause_sum[cause_mismatch]
    cause_review["unattributedMinutes"] = (
        cause_review["ArrDelayMinutes"] - cause_review["reportedCauseMinutes"]
    )
    print(f"\nUnattributed delayed flights: {len(cause_review):,}")
    print(f"Unattributed delay minutes: {cause_review['unattributedMinutes'].sum():,.0f}")
    print("Counts by reporting airline:")
    print(cause_review.groupby("Reporting_Airline").size().to_string())
    print("\nFirst 10 records requiring cause-attribution review:")
    print(cause_review.head(10).to_string(index=False))

## 5. Define the dashboard metrics explicitly

A completed flight is neither cancelled nor diverted and has a reported `ArrDelayMinutes`. A delayed flight is a completed flight with `ArrDel15 = 1`. The BTS on-time convention therefore means arrival less than 15 minutes late, not necessarily arrival at or before the scheduled time.

In [ ]:
delayed_15 = completed & df["ArrDel15"].eq(1)
metric_summary = pd.Series({
    "reported_flights": int(len(df)),
    "completed_flights": int(completed.sum()),
    "delayed_15_flights": int(delayed_15.sum()),
    "cancelled_flights": int(cancelled.sum()),
    "diverted_flights": int(diverted.sum()),
    "on_time_arrival_rate": 1 - delayed_15.sum() / completed.sum(),
    "average_delay_minutes_among_delayed_arrivals": df.loc[delayed_15, "ArrDelayMinutes"].sum() / delayed_15.sum(),
    "cancellation_rate": cancelled.sum() / len(df),
})
print(metric_summary.to_string())

In [ ]:
airline_profile = (
    pd.DataFrame({
        "airline": df["Reporting_Airline"],
        "flights": 1,
        "completed": completed.astype(int),
        "delayed_15": delayed_15.astype(int),
        "cancelled": cancelled.astype(int),
    })
    .groupby("airline", dropna=False)
    .sum()
)
airline_profile["on_time_rate"] = 1 - airline_profile["delayed_15"] / airline_profile["completed"]
airline_profile["cancellation_rate"] = airline_profile["cancelled"] / airline_profile["flights"]
print(airline_profile.sort_values("flights", ascending=False).round(4).to_string())

## 6. Aggregate for coordinated views

The browser does not need 631,970 flight-level rows. Records are aggregated by date, weekday, a two-hour scheduled departure window, reporting airline, and origin. All 352 origins remain individually selectable. This preserves the dimensions used by cross-filtering while reducing the download size.

In [ ]:
origin_counts = Counter(df["Origin"].dropna().tolist())
keep_origins = list(origin_counts) if TOP_AIRPORTS <= 0 else [code for code, _ in origin_counts.most_common(TOP_AIRPORTS)]
keep_origin_set = set(keep_origins)
origin_coverage = sum(origin_counts[code] for code in keep_origins) / len(df)
print(f"{len(keep_origins)} origins retain {origin_coverage:.1%} of flights as individually selectable airports.")
print(keep_origins)

In [ ]:
def departure_window(value):
    if pd.isna(value):
        return "Unknown"
    hhmm = int(float(value))
    hour = (hhmm // 100) % 24
    start = (hour // 2) * 2
    return f"{start:02d}-{(start + 2) % 24:02d}"

metric_frame = pd.DataFrame({
    "date": df["FlightDate"],
    "dayOfWeek": df["DayOfWeek"].astype("Int64"),
    "departureWindow": df["CRSDepTime"].map(departure_window),
    "airline": df["Reporting_Airline"],
    "origin": df["Origin"].where(df["Origin"].isin(keep_origin_set), "Other"),
    "flights": 1.0,
    "completed": completed.astype(float),
    "delayed15": delayed_15.astype(float),
    "cancelled": cancelled.astype(float),
    "diverted": diverted.astype(float),
    "delayedArrivalDelayMinutes": df["ArrDelayMinutes"].where(delayed_15, 0).fillna(0).astype(float),
    "carrierDelayMinutes": df["CarrierDelay"].fillna(0).astype(float),
    "weatherDelayMinutes": df["WeatherDelay"].fillna(0).astype(float),
    "nasDelayMinutes": df["NASDelay"].fillna(0).astype(float),
    "securityDelayMinutes": df["SecurityDelay"].fillna(0).astype(float),
    "lateAircraftDelayMinutes": df["LateAircraftDelay"].fillna(0).astype(float),
    "distanceMiles": df["Distance"].fillna(0).astype(float),
})

group_columns = ["date", "dayOfWeek", "departureWindow", "airline", "origin"]
metric_columns = [column for column in metric_frame.columns if column not in group_columns]
grouped = (
    metric_frame.groupby(group_columns, dropna=False, sort=True)[metric_columns]
    .sum()
    .reset_index()
)
print(f"Aggregate rows: {len(grouped):,}")
print(grouped.head(5).to_string(index=False))

## 7. Reconcile raw and aggregated totals

This is the most important audit. Every additive metric must have the same total before and after aggregation.

In [ ]:
raw_totals = metric_frame[metric_columns].sum()
aggregate_totals = grouped[metric_columns].sum()
reconciliation = pd.DataFrame({
    "raw_total": raw_totals,
    "aggregate_total": aggregate_totals,
})
reconciliation["difference"] = reconciliation["aggregate_total"] - reconciliation["raw_total"]
reconciliation["status"] = reconciliation["difference"].abs().le(0.01).map({True: "PASS", False: "FAIL"})
print(reconciliation.round(2).to_string())
assert reconciliation["status"].eq("PASS").all(), "Aggregation changed one or more totals."

## 8. Build and compare the dashboard payload

The payload is built in memory first. It is compared with the existing dashboard file before any overwrite is permitted.

In [ ]:
AIRLINE_NAMES = {
    "AA": "American Airlines", "AS": "Alaska Airlines",
    "B6": "JetBlue Airways", "DL": "Delta Air Lines",
    "F9": "Frontier Airlines", "G4": "Allegiant Air",
    "MQ": "Envoy Air", "OH": "PSA Airlines",
    "OO": "SkyWest Airlines", "UA": "United Airlines",
    "WN": "Southwest Airlines", "YX": "Republic Airways",
}

source_airlines = set(df["Reporting_Airline"].dropna().unique())
unmapped_airlines = sorted(source_airlines - set(AIRLINE_NAMES))
print(f"Unmapped airline codes: {unmapped_airlines or 'None'}")

origin_names = (
    df[["Origin", "OriginCityName"]]
    .dropna(subset=["Origin"])
    .drop_duplicates("Origin", keep="first")
    .set_index("Origin")["OriginCityName"]
    .to_dict()
)
schema = group_columns + metric_columns
compact_rows = []
for row in grouped.itertuples(index=False, name=None):
    keys = [row[0], int(row[1]), row[2], row[3], row[4]]
    values = [round(float(value), 2) for value in row[5:]]
    compact_rows.append(keys + values)

payload = {
    "meta": {
        "title": "BTS Reporting Carrier On-Time Performance",
        "period": "July 2026",
        "releaseDate": "2026-09-21",
        "sourceUrl": "https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGJ",
        "downloadUrl": "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_7.zip",
        "reportedFlights": int(len(df)),
        "originCount": len(origin_counts),
        "retainedOriginCount": len(keep_origin_set),
        "reportingCarrierCount": len(AIRLINE_NAMES),
        "aggregation": "Flight-level records aggregated by date, day of week, two-hour scheduled departure window, reporting carrier, and origin airport.",
        "metricDefinitions": {
            "onTimeRate": "1 - delayed arrivals (ArrDel15 = 1) / completed flights",
            "averageDelayMinutes": "Arrival delay minutes / delayed arrivals, limited to completed flights with ArrDel15 = 1",
            "cancellationRate": "Cancelled flights / reported flights",
        },
        "preparationNotes": [
            "Cancelled and diverted flights are excluded from arrival-performance denominators.",
            "Blank delay-cause values are treated as zero because BTS reports causes only when applicable.",
            "All origin airports are retained as individually selectable categories.",
            "Scheduled departure times are grouped into twelve two-hour local-time windows.",
        ],
    },
    "schema": schema,
    "airlines": AIRLINE_NAMES,
    "origins": {code: origin_names.get(code, code) for code in sorted(keep_origin_set)},
    "rows": compact_rows,
}

print(f"Payload rows: {len(payload['rows']):,}")
print(f"Payload schema fields: {len(payload['schema'])}")

In [ ]:
if OUTPUT_JSON.exists():
    existing_payload = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
    print(f"Existing dashboard rows: {len(existing_payload['rows']):,}")
    print(f"Schema match: {existing_payload['schema'] == payload['schema']}")
    print(f"Metadata match: {existing_payload['meta'] == payload['meta']}")
    print(f"Full semantic match: {existing_payload == payload}")
else:
    print("No existing dist/data.json was found; comparison skipped.")

## 9. Optional export

Leave the switch `False` while auditing. Change it to `True` only after reviewing all checks and reconciliation results. This makes overwriting the website data an explicit decision.

In [ ]:
WRITE_DASHBOARD_DATA = False

if WRITE_DASHBOARD_DATA:
    OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_JSON.write_text(
        json.dumps(payload, separators=(",", ":")), encoding="utf-8"
    )
    print(f"Wrote audited dashboard data to {OUTPUT_JSON}")
else:
    print("Audit-only mode: dist/data.json was not changed.")

## 10. Remaining limitations

Passing the checks above establishes internal consistency; it does not prove that BTS measurements are error-free or that the dashboard answers every possible question. Important limitations are:

- The data covers one month. It supports within-month comparisons, not seasonal or annual conclusions.
- `Reporting_Airline` represents the operating/reporting carrier, which may differ from the brand used to purchase a codeshare flight.
- `ArrDelayMinutes` sets early arrivals to zero. The displayed average uses only completed arrivals with `ArrDel15 = 1`; it is not a signed schedule deviation for all flights.
- Cancelled and diverted flights are excluded from the on-time denominator and shown separately. Changing that denominator would change airline rankings.
- Delay-cause fields are carrier-reported attribution measures and should not be interpreted as proof of causality.
- All 352 origin airports remain selectable, but the ranking view displays the 12 highest-volume origins in the current selection.
- Two-hour windows use scheduled local departure time, not actual departure time.
- BTS can revise previously released data. Record the download date and rerun this audit after replacing the source.
- The notebook checks consistency and coverage. Two-person usability testing is still required to determine whether people understand the views and interactions.